<a href="https://colab.research.google.com/github/oddzcv/yolov5/blob/master/YOLO_%2B_Attention_Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/RethinkingCoding/SMN-YOLOv5s

Cloning into 'SMN-YOLOv5s'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 203 (delta 1), reused 1 (delta 0), pack-reused 197 (from 1)
Receiving objects: 100% (203/203), 55.39 MiB | 14.96 MiB/s, done.
Resolving deltas: 100% (81/81), done.


In [2]:
%cd SMN-YOLOv5s
%pip install -qr requirements.txt

/content/SMN-YOLOv5s
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:00


In [3]:
import os

print(os.getcwd())

/content/SMN-YOLOv5s


In [4]:
import random

import numpy as np
import torch


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [5]:
!pip install roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="SYLND05UAeOcirwZfGFJ")
project = rf.workspace("potato-lxebf").project("potato-counting")
version = project.version(1)
dataset = version.download("yolov5")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to potato-counting-1 in yolov5pytorch:: 100%|██████████| 4838/4838 [00:04<00:00, 1090.83it/s]


In [9]:
!WANDB_MODE=disabled WANDB_DISABLED=true

In [11]:
!pip -q uninstall -y albumentations
!pip -q install "albumentations==1.3.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 8.0 MB/s eta 0:00:00


In [13]:
# =========================
# PATCH — disable deterministic algorithms that crash with AdaptiveMaxPool2d (CBAM)
# =========================
import re
from pathlib import Path

ROOT = Path("/content/SMN-YOLOv5s")  # adjust if your repo path differs

py_files = list(ROOT.rglob("*.py"))
patched = []

for p in py_files:
    txt = p.read_text(errors="ignore")

    new = txt

    # 1) Disable strict deterministic algorithms
    new = re.sub(r"torch\.use_deterministic_algorithms\(\s*True\s*\)", "torch.use_deterministic_algorithms(False)", new)

    # 2) If init_seeds(..., deterministic=True) exists, flip it
    new = re.sub(r"(init_seeds\([^)]*?)deterministic\s*=\s*True", r"\1deterministic=False", new)

    if new != txt:
        p.write_text(new)
        patched.append(str(p))

print(f"Patched {len(patched)} file(s).")
for f in patched[:20]:
    print(" -", f)

Patched 4 file(s).
 - /content/SMN-YOLOv5s/train.py
 - /content/SMN-YOLOv5s/utils/general.py
 - /content/SMN-YOLOv5s/segment/train.py
 - /content/SMN-YOLOv5s/classify/train.py


In [16]:
!python train.py --img 640 --batch 16 --epochs 10 --data {dataset.location}/data.yaml --weights '' --cfg /content/SMN-YOLOv5s/models/yolov5s-cbam.yaml --cache

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-02-15 09:10:02.901536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771146602.922334    8916 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771146602.929322    8916 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771146602.948074    8916 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771146602.948102    8916 computation_placer.cc:177] computation placer already registere

In [17]:
!python train.py --img 640 --batch 16 --epochs 10 --data {dataset.location}/data.yaml --weights '' --cfg /content/SMN-YOLOv5s/models/yolov5s-c3se.yaml --cache

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-02-15 09:23:11.834528: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771147391.855926   12413 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771147391.862975   12413 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771147391.880929   12413 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771147391.880954   12413 computation_placer.cc:177] computation placer already registere

In [19]:
!python detect.py --weights runs/train/exp8/weights/best.pt --img 640 --conf 0.1 --source {dataset.location}/test/images

detect: weights=['runs/train/exp8/weights/best.pt'], source=/content/SMN-YOLOv5s/potato-counting-1/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.1, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 704b81b1 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5s-cbam summary: 224 layers, 7074200 parameters, 0 gradients, 15.8 GFLOPs
image 1/168 /content/SMN-YOLOv5s/potato-counting-1/test/images/80_100_0-25_0001_jpg.rf.e6eb44d08cce17b3d731fc3289476049.jpg: 384x640 24 kentangs, 58.4ms
image 2/168 /content/SMN-YOLOv5s/potato-counting-1/test/images/80_100_0-25_0023_jpg.rf.a6688cb1ed6fcb550d42248530abba77.jpg: 384x640 19 kentangs, 18

In [20]:
!python detect.py --weights runs/train/exp9/weights/best.pt --img 640 --conf 0.1 --source {dataset.location}/test/images

detect: weights=['runs/train/exp9/weights/best.pt'], source=/content/SMN-YOLOv5s/potato-counting-1/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.1, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 704b81b1 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5s-c3se summary: 283 layers, 6735734 parameters, 0 gradients, 15.2 GFLOPs
image 1/168 /content/SMN-YOLOv5s/potato-counting-1/test/images/80_100_0-25_0001_jpg.rf.e6eb44d08cce17b3d731fc3289476049.jpg: 384x640 31 kentangs, 45.1ms
image 2/168 /content/SMN-YOLOv5s/potato-counting-1/test/images/80_100_0-25_0023_jpg.rf.a6688cb1ed6fcb550d42248530abba77.jpg: 384x640 33 kentangs, 12

In [21]:
from IPython.display import Image

In [23]:
Image("/kaggle/working/SMN-YOLOv5s/runs/train/exp8/confusion_matrix.png")

FileNotFoundError: No such file or directory: '/kaggle/working/SMN-YOLOv5s/runs/train/exp8/confusion_matrix.png'

FileNotFoundError: No such file or directory: '/kaggle/working/SMN-YOLOv5s/runs/train/exp8/confusion_matrix.png'

<IPython.core.display.Image object>